# Fourier regression: data → spectral initialization → Adam → saved-model inference

Run cells from top to bottom using this repository's `.venv` Python kernel.
The default is **1,000 in-domain points** (700 train / 100 validation / 200 test),
K=8, and seed 42. Each execution creates a fresh timestamped output directory.
The notebook calls the existing project pipeline rather than implementing a second trainer.

If a notebook interface is not installed, install JupyterLab separately in your environment.
In VS Code, select `.venv/bin/python` as the notebook kernel.


In [ ]:
from pathlib import Path
import os
import sys

# # Locate the repository whether the kernel starts here or in notebooks/.
# ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
#              if (p / "pyproject.toml").is_file()
#              and (p / "function_approximation").is_dir()), None)
# if ROOT is None:
#     raise RuntimeError("Start this notebook from the repository or notebooks/ directory.")
# sys.path.insert(0, str(ROOT))
# os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs" / ".matplotlib"))

import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from function_approximation.data.generation import DataGenerator
from function_approximation.data.preprocessing import DataPreprocessor
from function_approximation.models import build_model
from function_approximation.training.config import load_experiment_config
from function_approximation.evaluation.metrics import compute_metrics
from function_approximation.inference import load_predictor
from scripts.run_experiments import run_experiment, set_seed

torch.set_num_threads(1)
# print("Repository:", ROOT)
# print("Python:", sys.executable)
# print("PyTorch:", torch.__version__)


## 1. Configure the experiment

Change `SAMPLE_SIZE` to 10,000 to restore the original sample budget.
The 2,000-point near/far evaluation grids are independent of this budget.
The [0.1, 20] frequency interval is an explicit search prior, not supplied target frequencies.


In [ ]:
SAMPLE_SIZE = 1000
SEED = 42
K = 8

config = load_experiment_config(ROOT / "configs/experiments/fourier_rich_periodic.yaml")
config["data"]["sample_size"] = SAMPLE_SIZE
config["model"]["num_frequencies"] = K
config["experiment"]["seed"] = SEED
config["experiment"]["run_name"] = f"notebook_fourier_k{K}_n{SAMPLE_SIZE}_seed{SEED}"
config["tracking"]["enabled"] = False
session_dir = ROOT / "outputs/notebooks/fourier" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
session_dir.mkdir(parents=True, exist_ok=False)
config["experiment"]["output_dir"] = (session_dir / "runs").relative_to(ROOT).as_posix()
config_path = session_dir / "config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
config = load_experiment_config(config_path)
print(yaml.safe_dump(config, sort_keys=False))


## 2. Generate data and check the splits

Only training pairs enter spectral fitting and gradient updates. Validation selects
the checkpoint; interpolation test and near/far labels are for reporting only.


In [ ]:
generator = DataGenerator(**config["data"])
dataset = generator.generate()
bands = generator.generate_extrapolation()
evaluation_data = {**dataset, **bands}
display(pd.DataFrame([
    {"split": name, "samples": len(x), "min_x / pi": float(np.min(x) / np.pi),
     "max_x / pi": float(np.max(x) / np.pi)}
    for name, (x, y) in evaluation_data.items()
]))
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    assert np.intersect1d(dataset[left][0].astype(np.float32),
                          dataset[right][0].astype(np.float32)).size == 0
print("In-domain splits are disjoint after float32 conversion.")


## 3. Inspect the spectral starting fit

This preview uses training data only. The initializer searches residuals for
Lomb–Scargle peaks, refines each new frequency locally, and jointly refits all
selected sine/cosine coefficients by least squares. It is a greedy initialization,
not a guarantee of recovering the globally best frequencies.

The preview is not the saved final model. The runner below repeats initialization
with the same seed/data before Adam, keeping training and artifact saving in one place.


In [ ]:
preprocessor = DataPreprocessor(**config["preprocessing"])
processed = preprocessor.fit_transform_dataset(dataset)
set_seed(SEED)
preview_model = build_model(config["model"]["type"],
                            {k: v for k, v in config["model"].items() if k != "type"})
initialization = preview_model.initialize_from_data(*processed["train"])
print("Active components:", initialization["active_components"])
print("Float64 least-squares training MSE:", initialization["least_squares_train_mse"])
display(pd.DataFrame(initialization["spectrum"]["components"]))
with torch.inference_mode():
    initial_prediction = preview_model(torch.from_numpy(processed["train"][0])).numpy()
print("PyTorch initial training RMSE:", compute_metrics(dataset["train"][1], initial_prediction)["rmse"])


## 4. Train with Adam and save the selected checkpoint

The full configured budget is up to 30,000 epochs, with patience 3,000 and
`min_delta=1e-12` in validation MSE units. Adam updates frequencies, weights, and
bias together; its frequency updates are not restricted to the initialization bounds.
The runner regenerates the same deterministic data and saves the resolved config,
checkpoint, preprocessor, predictions, metrics, diagnostics, and plots.


In [ ]:
run_dir = run_experiment(config_path)
print("Saved run:", run_dir)


## 5. Evaluate the selected model and inspect its plots

These scores concern one noise-free periodic target and one seed. To establish
which configuration is best, run the separate initialization/capacity ablation;
this notebook trains one chosen spectral + Adam recipe.


In [ ]:
metrics = json.loads((run_dir / "metrics/metrics.json").read_text())
baseline = json.loads((run_dir / "metrics/baseline_metrics.json").read_text())
display(pd.DataFrame(metrics).T)
print("Training-mean baseline:", baseline["prediction"])
for plot_name in ("learning_curves.png", "extrapolation.png"):
    display(Image(filename=str(run_dir / "plots" / plot_name)))


## 6. Reload the saved model for inference

The inference loader needs only the run's model artifacts. Pass raw input coordinates;
it applies saved preprocessing automatically. Prediction does not call the target
function or need labels. The points below include inputs outside the training window.

To load a run from an earlier session, replace `INFERENCE_RUN` with its run-root path.


In [ ]:
import matplotlib.pyplot as plt
from function_approximation.evaluation.visualization import (
    GROUND_TRUTH, PREDICTION, pi_axis, style_figure,
)

INFERENCE_RUN = run_dir
predictor = load_predictor(INFERENCE_RUN)

# New inference inputs beyond the training interval [-10π, 10π].
x_new = np.linspace(1200 * np.pi, 1250 * np.pi, 1000)

# Model inference does not use the true function.
y_pred = predictor.predict(x_new)

# Reference values are used only to evaluate and visualize predictions.
y_true = generator.target_fn(x_new)

display(pd.DataFrame({
    "x / pi": x_new / np.pi,
    "true value": y_true,
    "prediction": y_pred,
    "absolute error": np.abs(y_true - y_pred),
}))

scores = compute_metrics(y_true, y_pred)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    x_new, y_true,
    color=GROUND_TRUTH, linewidth=4, label="True value",
)
ax.plot(
    x_new, y_pred,
    color=PREDICTION, linewidth=2, linestyle="--",
    label="Model prediction",
)
ax.set(
    title=f"Inference on new inputs · RMSE {scores['rmse']:.3e}",
    xlabel="Input x",
    ylabel="f(x)",
)
pi_axis(ax, x_new.min(), x_new.max(), ticks=5)
ax.legend(frameon=False)
style_figure(fig)
plt.show()

## 7. Verify checkpoint reload against saved predictions

This final check compares inference outputs to the runner's saved export and
recomputes the test score. Reference labels are read only for scoring here.


In [ ]:
with np.load(run_dir / "data/predictions.npz") as saved:
    for split in metrics:
        reloaded = predictor.predict(saved[f"{split}_x"])
        np.testing.assert_array_equal(reloaded, saved[f"{split}_pred"])
    test_metrics = compute_metrics(saved["test_y"], predictor.predict(saved["test_x"]))
assert np.isclose(test_metrics["rmse"], metrics["test"]["rmse"], rtol=1e-10, atol=1e-12)
print("All saved split predictions match the reloaded checkpoint exactly.")
print("Verified test RMSE:", test_metrics["rmse"])
print("CLI equivalent:")
print(f'python -m function_approximation.inference --run-dir "{run_dir}" --x 0 100')
